# Reproducing Nishimoto et al. (2011) from CRCNS vim-2

**Paper.** Nishimoto, Vu, Naselaris, Benjamini, Yu & Gallant (2011), "Reconstructing Visual Experiences from Brain Activity Evoked by Natural Movies", Current Biology 21, 1641-1646, doi:10.1016/j.cub.2011.08.031.

**Claims file.** `claims/nishimoto-2011-reconstructing.yaml`, written from the version of record (main text, legends, Experimental Procedures) and its Supplemental Information.

**Dataset.** CRCNS vim-2 (Gallant Lab Natural Movie 4T fMRI Data), doi:10.6080/K00Z715X, read from the public S3 bucket `crcnsarchive` under `vim-2/`. CRCNS data sets are not versioned, so the version is recorded as `unversioned` (docs/archives.md); there is only one published version to pin, and it is the one the data descriptor (version 0.61, 2015-02-17) describes as holding the data of this paper. The MD5 sums of the files read, from the data set's `checksums.md5`, are printed in the data check below so that a later change can be detected: Stimuli.tar.gz d06069177dd2ccad303009b7003c67b9, VoxelResponses_subject1.tar.gz 0e389bfa0b2235d1763d8a0d3bebf0af, VoxelResponses_subject2.tar.gz 92f3e621677ac1c8c226c87d77977c88, VoxelResponses_subject3.tar.gz 3ea1084138f70583228ad5649c935921. The raw per-session data (`raw_data/`) are not used: the paper's analysis starts from the preprocessed responses these files hold.

Each file is a tar.gz holding one MATLAB 7.3 (HDF5) file. It is streamed once from the bucket into memory and opened there (vim2.py); nothing is written to disk.

| Claim | Tested here |
|---|---|
| fig2de-prediction-accuracy | tested |
| fig2e-directional-gain-small | tested |
| figs1a-motion-better-every-area | tested |
| fig2fgh-retinotopic-receptive-fields | tested (proxy test of the claims file, without the multifocal maps) |
| fig2j-speed-eccentricity | tested |
| fig3ab-identification-s1 | tested |
| fig3c-identification-large-set | blocked, data_missing (1,000,000-clip set not public) |
| fig4-reconstructions | blocked, data_missing (sampled prior not public) |
| fig4d-reconstruction-accuracy | blocked, data_missing (sampled prior not public) |
| discussion-color-no-gain | tested in the second notebook, `color_no_gain.ipynb` |

**Main deviation.** The paper fits each voxel with L1-regularized least squares by coordinate descent (Model Fitting, refs. 15 and 16). Every model here is fit by ridge regression, solved through the kernel of the training features, with the penalty chosen per voxel on held-out training data, under both readings of the claims file: (a) a held-out 10% in 50 s blocks, (b) 5-fold cross-validation over such blocks. This is an assumption: the paper gives neither its L1 penalty nor its stopping rule, and ridge with a penalty per voxel is the fit the same laboratory used for later encoding models of the same kind. An L1 fit by coordinate descent on the Gram matrix of the 26,220 directional regressors was tried while preparing this revision, on 40 voxels of S1, and did not converge within 1,000 sweeps per penalty in 30 minutes; it is not run here.

In [ ]:
import time, resource, collections, urllib.request
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import vim2

T0 = time.time()
SEED = 2011  # standard choice; each claim's random draws use their own generator
ALPHAS = np.logspace(0, 9, 19).astype(np.float32)  # ridge penalties searched per voxel
MODELS = ["static", "nondirectional", "directional"]
SUBJECTS = [1, 2, 3]
def mem_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6

## Stimuli and motion-energy features

Frames are downsampled to 96 x 96, converted to CIE L*A*B*, and the luminance passes through the 6,555-filter motion-energy bank (Supplemental, Motion-Energy Encoding Model; Gabor Wavelet Basis Set), one 600 s run (or 60 s test block) at a time. The bank is pymoten's pyramid; its default envelopes give the paper's count of 6,555 filters (see vim2.py). Four training frames are shown to check the orientation of the stored frames (rows and columns).

In [ ]:
t = time.time()
filters = vim2.pyramid().filters
EXAMPLES = (0, 30000, 60000, 90000)
S = vim2.stimulus_motion_energy(filters, ("L",), example_frames=EXAMPLES)
me_train, me_test = S["st"]["L"], S["sv"]["L"]
stim_info = {k: S[k] for k in ("member", "member_bytes", "layout", "st_range", "sv_range", "st_L_range", "sv_L_range")}
examples = S["examples"]
del S
print(f"stimuli streamed, converted and filtered in {time.time() - t:.0f} s; peak memory {mem_gb():.1f} GB")
fig, ax = plt.subplots(1, 4, figsize=(10, 2.8))
for a, i in zip(ax, EXAMPLES):
    a.imshow(examples[i])
    a.set_title(f"training frame {i}", fontsize=9)
    a.axis("off")
plt.show()

In [ ]:
md5 = urllib.request.urlopen(vim2.BASE + "checksums.md5", timeout=60).read().decode()
print("MD5 sums of the files read (checksums.md5):")
for line in md5.splitlines():
    if "Stimuli" in line or "VoxelResponses" in line:
        print(" ", line)
print("Stimulus member:", stim_info["member"], stim_info["member_bytes"], "bytes")
print("Stored arrays:", stim_info["layout"])
print("Training frames", stim_info["layout"]["st"][0][0], "= 7200 s x 15 Hz:", stim_info["layout"]["st"][0][0] == 7200 * 15)
print("Test frames", stim_info["layout"]["sv"][0][0], "= 540 s x 15 Hz:", stim_info["layout"]["sv"][0][0] == 540 * 15)
print("Pixel range (uint8), training", stim_info["st_range"], "test", stim_info["sv_range"])
print("Luminance L* range after 96 x 96 downsampling, training %.1f to %.1f, test %.1f to %.1f (0 to 100 expected)"
      % tuple(stim_info["st_L_range"] + stim_info["sv_L_range"]))
print("Filters in the bank:", len(filters), "(paper: 6,555)")
c = collections.Counter((float(f["spatial_freq"]), float(f["temporal_freq"]), float(f["direction"])) for f in filters)
print("Filters per (spatial frequency c/image, temporal frequency Hz): count, directions")
for sf in sorted({k[0] for k in c}):
    for tf in sorted({k[1] for k in c}):
        dirs = sorted(k[2] for k in c if k[:2] == (sf, tf))
        print(f"   ({sf:g}, {tf:g}): {sum(c[(sf, tf, d)] for d in dirs)} filters, {c[(sf, tf, dirs[0])]} per direction, directions {[int(d) for d in dirs]}")
z = collections.Counter(round(float(f["spatial_env"]), 4) for f in filters if f["spatial_freq"] == 0 and f["temporal_freq"] == 0)
print("Zero-spatial-frequency filters at each temporal frequency, by spatial envelope s.d. (fraction of the image):", dict(sorted(z.items())),
      "- one per position of the grid of each nonzero spatial frequency, with that frequency's envelope, and no direction")
for m in MODELS:
    cols = vim2.model_columns(filters, m)
    g = collections.Counter((float(filters[cc[0]]["spatial_freq"]), float(filters[cc[0]]["temporal_freq"]), len(cc)) for cc in cols)
    print(f"{m}: {len(cols)} features; (spatial freq, temporal freq, filters summed): count", dict(sorted(g.items())))
ktr = vim2.keep_mask(7200, vim2.TRAIN_BLOCK)
kte = vim2.keep_mask(540, vim2.TEST_BLOCK)
print("Motion energy at 1 Hz, training", me_train.shape, "test", me_test.shape,
      "range %.2f to %.2f (log units)" % (me_train.min(), me_train.max()))
print("Samples kept after discarding the first 6 s of each block: training", ktr.sum(), "(paper 7,128), test", kte.sum(), "(paper 486)")

## Features, kernels and splits

Each model's features are summed per column, z-scored with the training statistics, clipped at 3 and delayed by 3 to 6 s within runs. The ridge fits go through the kernel X X' of the training samples, which depends only on the features, so it is computed once per model and serves the three subjects. The held-out 10% (reading a of the penalty and the voxel selection of identification) and the five folds (reading b) are drawn once and used for every subject (standard choice).

In [ ]:
held = vim2.holdout_blocks(int(ktr.sum()), np.random.default_rng([SEED, 0]))
fold = vim2.cv_folds(int(ktr.sum()), np.random.default_rng([SEED, 1]))
KER, FEAT = {}, {}
t = time.time()
for m in MODELS + ["delay0", "delay4", "delay8"]:
    base = m if m in MODELS else "directional"
    dl = vim2.DELAYS if m in MODELS else (int(m[5:]),)
    cols = vim2.model_columns(filters, base)
    Xtr, Xte, mu, sd = vim2.build_features(me_train, me_test, cols, return_stats=True)
    X = vim2.delayed(Xtr, dl, 600)[ktr]
    Xe = vim2.delayed(Xte, dl, 60)[kte]
    KER[m] = (X @ X.T, Xe @ X.T)
    if m == "directional":
        FEAT[m] = dict(X=X, cols=cols, mu=mu, sd=sd)
    print(f"{m}: {len(cols)} features x {len(dl)} delays = {X.shape[1]} regressors")
    del X, Xe, Xtr, Xte
print(f"held-out samples {held.sum()} of {ktr.sum()}; fold sizes {np.bincount(fold).tolist()}; kernels in {time.time() - t:.0f} s")

## Responses, per subject

For each subject the V1, V2, V3, V3A and V3B voxels are taken from the data set's ROI masks. Voxels with any missing value are dropped and counted. The test responses are the ten repeats (`rva`); following Data Preprocessing ("signals were then normalized to mean 0.0 and standard deviation 1.0" for each run), each repeat is z-scored per voxel, and the observed test signal of reading (a) is their mean. The stored repeat mean `rv` is also scored, as a check.

In [ ]:
DATA = {}
for s in SUBJECTS:
    t = time.time()
    d = vim2.load_subject(s)
    good = ~(np.isnan(d["rt"]).any(0) | np.isnan(d["rva"]).any((0, 1)) | np.isnan(d["rv"]).any(0))
    rva = d["rva"][:, :, good]
    raw_std = (float(d["rt"][:, good].std()), float(d["rv"][:, good].std()), float(rva.std()))
    rv_check = float(np.max(np.abs(d["rv"][:, good] - rva.mean(0))))
    rva = (rva - rva.mean(1, keepdims=True)) / rva.std(1, keepdims=True)
    Yrep = rva[:, kte]
    DATA[s] = dict(Y=d["rt"][ktr][:, good], Yrep=Yrep, Yte=Yrep.mean(0), Yrv=d["rv"][kte][:, good],
                   vox=d["vox"][good], area=d["area"][good], hemi=d["hemi"][good],
                   info=dict(layout=d["layout"], ei=d["ei"], roi_values=d["roi_values"], roi_counts=d["roi_counts"],
                             roi_mean_slice=d["roi_mean_slice"], n_any=d["n_voxels_any_roi"], n_multi=d["n_multi_label"],
                             n_roi=len(d["vox"]), n_drop=int((~good).sum()), raw_std=raw_std, rv_check=rv_check,
                             first6=float(d["rt"][~ktr][:, good].mean()), rest=float(d["rt"][ktr][:, good].mean())))
    del d, rva
    print(f"subject {s}: loaded in {time.time() - t:.0f} s; peak memory {mem_gb():.1f} GB")

## Fits

For each model and each split the training block of the kernel is decomposed once; the penalty of each voxel is the one whose fit best predicts the held-out samples (reading a: the held-out 10%, over the 19 penalties of the grid; reading b: the mean correlation over the five folds, over the whole decades of the grid, for time), and the model is refit on all training samples with it. The clock check fits the directional model at a single delay of 0, 4 or 8 s: if responses and stimuli share a clock, the 4 s delay (the hemodynamic lag the paper assumes) predicts best and 0 s worst.

In [ ]:
FIT = {s: {} for s in SUBJECTS}
idx_all = np.arange(int(ktr.sum()))
for m in MODELS + ["delay0", "delay4", "delay8"]:
    t = time.time()
    K, Kte = KER[m]
    tr, ev = idx_all[~held], idx_all[held]
    w, U = vim2.eig_split(K, tr)
    Kev = K[np.ix_(ev, tr)]
    for s in SUBJECTS:
        Y = DATA[s]["Y"]
        rh = vim2.ridge_path_corr(w, U, Kev, Y[tr], Y[ev], ALPHAS)
        FIT[s][m] = dict(ai_a=rh.argmax(0), held_acc=rh.max(0))
        if m in ("directional", "delay4") and s == 1:
            C90 = vim2.ridge_dual(w, U, Y[tr], FIT[s][m]["ai_a"], ALPHAS)
            FIT[s][m]["Pheld"] = Kev @ C90
            FIT[s][m]["R90"] = Y[tr] - K[np.ix_(tr, tr)] @ C90
            del C90
    if m in MODELS:
        cv = {s: 0 for s in SUBJECTS}
        for k in range(5):
            tr_k, ev_k = idx_all[fold != k], idx_all[fold == k]
            wk, Uk = vim2.eig_split(K, tr_k)
            Kk = K[np.ix_(ev_k, tr_k)]
            for s in SUBJECTS:
                Y = DATA[s]["Y"]
                cv[s] = cv[s] + vim2.ridge_path_corr(wk, Uk, Kk, Y[tr_k], Y[ev_k], ALPHAS[::2]) / 5
            del wk, Uk, Kk
        for s in SUBJECTS:
            FIT[s][m]["ai_b"] = 2 * cv[s].argmax(0)  # whole decades of the same grid, for time
    w, U = vim2.eig_split(K, idx_all)
    for s in SUBJECTS:
        Y, f = DATA[s]["Y"], FIT[s][m]
        for rd in ("a", "b") if m in MODELS else ("a",):
            C = vim2.ridge_dual(w, U, Y, f["ai_" + rd], ALPHAS)
            P = Kte @ C
            f["acc_" + rd] = {"avg": vim2.corr_cols(P, DATA[s]["Yte"]),
                              "rep": np.mean([vim2.corr_cols(P, DATA[s]["Yrep"][k]) for k in range(10)], 0),
                              "rv": vim2.corr_cols(P, DATA[s]["Yrv"])}
            if rd == "a" and m in ("directional", "delay4"):
                f["Pte"] = P
                if m == "directional":
                    f["C"] = C
                if s == 1:
                    f["R"] = Y - K @ C
            del C, P
    del w, U, Kev
    print(f"{m}: fitted in {time.time() - t:.0f} s; peak memory {mem_gb():.1f} GB")

In [ ]:
for s in SUBJECTS:
    i = DATA[s]["info"]
    print(f"--- subject {s}")
    print("  stored arrays:", i["layout"], "ei:", i["ei"])
    print("  ROI mask values:", i["roi_values"], "; voxels in any ROI:", i["n_any"], "of 73,728")
    print("  V1-V3B voxels per mask:", {k: v for k, v in i["roi_counts"].items() if k[:2] in ("v1", "v2", "v3")})
    print("  voxels in more than one V1-V3B mask:", i["n_multi"])
    print(f"  V1-V3B voxels {i['n_roi']} (paper about 4,500 per subject); dropped for missing values {i['n_drop']}; kept {i['n_roi'] - i['n_drop']}")
    print("  kept per area:", dict(collections.Counter(DATA[s]["area"])))
    print(f"  rv equals the mean of the ten rva repeats: max abs difference {i['rv_check']:.2e}")
    print(f"  s.d. of rt {i['raw_std'][0]:.3f}, rv {i['raw_std'][1]:.3f}, rva {i['raw_std'][2]:.3f} (training z-scored per run; test in other units, so each repeat is z-scored here)")
    print(f"  mean training response in the first 6 s of each run {i['first6']:.3f}, in the kept samples {i['rest']:.3f} (onset transient, discarded as in the paper)")
    for m in MODELS:
        for rd in ("a", "b"):
            n = np.bincount(FIT[s][m]["ai_" + rd], minlength=len(ALPHAS))
            print(f"  {m}, penalty reading ({rd}): voxels per chosen penalty (1 to 1e9, half decades) {n.tolist()}; at the lowest {n[0]}, at the highest {n[-1]}")
    print("  clock check, mean test accuracy of the directional model at one delay:",
          {f"{k[5:]} s": round(float(FIT[s][k]["acc_a"]["avg"].mean()), 3) for k in ("delay0", "delay4", "delay8")})

## fig2de-prediction-accuracy

> Averaged over subjects and voxels in V1, V2, V3, V3A and V3B, the prediction accuracy on the test data is 0.24 for the static model, 0.39 for the nondirectional motion-energy model and 0.40 for the directional motion-energy model, and the differences are significant (Wilcoxon signed-rank test, p < 0.0001).

Readings: accuracy on (a) the repeat-averaged test signal or (b) the mean over repeats of the single-repeat correlation; subjects combined by (i) pooling voxels or (ii) the mean of the three subject means; penalty chosen on (a) the held-out 10% or (b) five folds. Tolerances 0.024, 0.039, 0.040. The Wilcoxon tests are two-sided.

In [ ]:
EXP = {"static": (0.24, 0.024), "nondirectional": (0.39, 0.039), "directional": (0.40, 0.040)}
def acc(s, m, rd, pen="a"):
    return FIT[s][m]["acc_" + pen][rd]
def pooled(m, rd, pen="a"):
    return np.concatenate([acc(s, m, rd, pen) for s in SUBJECTS])
fig2de = {}
for pen in ("a", "b"):
    for rd in ("avg", "rep", "rv"):
        p_ds = stats.wilcoxon(pooled("directional", rd, pen), pooled("static", rd, pen)).pvalue
        p_dn = stats.wilcoxon(pooled("directional", rd, pen), pooled("nondirectional", rd, pen)).pvalue
        for comb in ("pooled", "subject means"):
            if comb == "pooled":
                means = {m: float(pooled(m, rd, pen).mean()) for m in MODELS}
            else:
                means = {m: float(np.mean([acc(s, m, rd, pen).mean() for s in SUBJECTS])) for m in MODELS}
            within = all(abs(means[m] - EXP[m][0]) <= EXP[m][1] for m in MODELS)
            order = means["static"] < means["nondirectional"] < means["directional"]
            fig2de[(pen, rd, comb)] = dict(means=means, within=within, order=order, p_ds=p_ds, p_dn=p_dn)
            lab = {"avg": "(a) repeat mean", "rep": "(b) single repeats", "rv": "check: stored rv"}[rd]
            print(f"penalty ({pen}), signal {lab}, ({'i' if comb == 'pooled' else 'ii'}) {comb}: "
                  + ", ".join(f"{m} {means[m]:.3f}" for m in MODELS)
                  + f"; within tolerance {within}; order {order}; Wilcoxon dir vs static p = {p_ds:.1e}, dir vs nondir p = {p_dn:.1e}")
    print()
print("Distance of the static mean from the lower bound 0.216, reading (a) signal, (i) pooled:",
      {pen: round(fig2de[(pen, "avg", "pooled")]["means"]["static"] - 0.216, 4) for pen in ("a", "b")})
print("Per subject, penalty (a), signal (a):")
for s in SUBJECTS:
    ps = stats.wilcoxon(acc(s, "directional", "avg"), acc(s, "static", "avg")).pvalue
    pn = stats.wilcoxon(acc(s, "directional", "avg"), acc(s, "nondirectional", "avg")).pvalue
    print(f"  S{s} (n = {len(acc(s, 'static', 'avg'))} voxels): " + ", ".join(f"{m} {acc(s, m, 'avg').mean():.3f}" for m in MODELS)
          + f"; p dir-static {ps:.1e}, dir-nondir {pn:.1e}")
print("Null reference: a voxel's correlation on 486 samples exceeds this by chance with p < 0.01 (two-sided):",
      round(float(stats.norm.ppf(0.995) / np.sqrt(486 - 3)), 3))
print("Range of voxel accuracies (penalty a, signal a, pooled):", {m: (round(float(pooled(m, 'avg').min()), 3), round(float(pooled(m, 'avg').max()), 3)) for m in MODELS})

**Result: reproduced, under penalty reading (b), resting on ridge regression in place of L1.** With the penalty chosen by five-fold cross-validation (penalty reading b) and the repeat-averaged test signal (signal reading a), the pooled means are 0.233 (static), 0.384 (nondirectional) and 0.386 (directional), each within its tolerance of 0.24, 0.39 and 0.40, in the paper's order, with both paired Wilcoxon tests at p below 0.0001 (directional against nondirectional p = 1.3e-22; against static printed as 0). The mean of subject means gives 0.234, 0.384 and 0.386. With the penalty chosen on the single held-out 10% (penalty reading a) the means are 0.211, 0.370 and 0.373: order and tests hold, but the static mean is 0.005 below its lower bound of 0.216, which alone would be partially reproduced. Single repeats (signal reading b) give about half (0.118 to 0.128 static, 0.207 to 0.212 directional), outside tolerance under both penalty readings. The stored repeat mean `rv` gives the same values as the z-scored repeats to within 0.003. The outcome rests on two choices the paper leaves open: ridge in place of L1, and the penalty reading.

## fig2e-directional-gain-small

> The directional model predicts better than the nondirectional model, but the difference is small, which suggests that BOLD signals carry spatially localized but mostly nondirectional motion information.

Reproduced when the directional mean exceeds the nondirectional mean, the paired Wilcoxon test gives p < 0.0001, and the difference is less than a tenth of the gain of the nondirectional model over the static model.

In [ ]:
for (pen, rd, comb), v in fig2de.items():
    mm = v["means"]
    gain_dir = mm["directional"] - mm["nondirectional"]
    gain_motion = mm["nondirectional"] - mm["static"]
    ok = gain_dir > 0 and v["p_dn"] < 1e-4 and gain_dir < 0.1 * gain_motion
    print(f"penalty ({pen}), signal {rd}, {comb}: directional - nondirectional = {gain_dir:.4f}; nondirectional - static = {gain_motion:.4f}; "
          f"ratio {gain_dir / gain_motion:.3f}; Wilcoxon p = {v['p_dn']:.1e}; criterion met {ok}")
for s in SUBJECTS:
    dd = acc(s, "directional", "avg") - acc(s, "nondirectional", "avg")
    print(f"  S{s}, penalty (a), signal (a): mean difference {dd.mean():.4f}, fraction of voxels with directional better {np.mean(dd > 0):.2f}")

**Result: reproduced.** Directional minus nondirectional accuracy is 0.0010 to 0.0034 against a nondirectional gain over static of 0.083 to 0.159, ratios of 0.012 to 0.021, all under the bound of 0.1, with Wilcoxon p below 0.0001 (largest 5.3e-22), under every reading of penalty, test signal and subject combination. The directional model is better in 53%, 62% and 55% of voxels of S1, S2 and S3 (penalty and signal reading a).

## figs1a-motion-better-every-area

> In each of the five visual areas and each of the three subjects, both motion models predict better than the static model, and the best predictions are in V1 for all three subjects (Wilcoxon rank-sum test, p < 0.0001).

Mean accuracy per model, area and subject (penalty reading a; signal reading a, with b shown too). Rank-sum readings: (a) V1 against each other area in turn; (b) V1 against the other four pooled. Model readings: (i) directional; (ii) each of the three models. The rank-sum p-values are given one-sided (V1 greater, as the claim is that V1 is best) and two-sided; the pass counts are given for both.

In [ ]:
AREAS = vim2.AREAS
for rd in ("avg", "rep"):
    print(f"=== test-signal reading {rd}")
    n_cells = n_beat = 0
    v1_best = {}
    for s in SUBJECTS:
        area = DATA[s]["area"]
        for ar in AREAS:
            sel = area == ar
            mm = {m: acc(s, m, rd)[sel].mean() for m in MODELS}
            beat = mm["nondirectional"] > mm["static"] and mm["directional"] > mm["static"]
            n_cells += 1
            n_beat += beat
            print(f"  S{s} {ar:4s} n={sel.sum():4d}: " + ", ".join(f"{m} {mm[m]:.3f}" for m in MODELS) + f"; both motion > static {beat}")
        for m in MODELS:
            a = acc(s, m, rd)
            means = {ar: a[area == ar].mean() for ar in AREAS}
            best = max(means, key=means.get)
            v1 = a[area == "v1"]
            p1 = {ar: stats.ranksums(v1, a[area == ar], alternative="greater").pvalue for ar in AREAS[1:]}
            p2 = {ar: stats.ranksums(v1, a[area == ar]).pvalue for ar in AREAS[1:]}
            q1 = stats.ranksums(v1, a[area != "v1"], alternative="greater").pvalue
            q2 = stats.ranksums(v1, a[area != "v1"]).pvalue
            v1_best[(s, m)] = (best == "v1", max(p1.values()) < 1e-4, q1 < 1e-4, max(p2.values()) < 1e-4, q2 < 1e-4)
            print(f"    S{s} {m}: best area {best}; rank-sum V1 > each other area, largest p one-sided {max(p1.values()):.1e}, two-sided {max(p2.values()):.1e} "
                  f"({ {k: f'{v:.0e}' for k, v in p1.items()} }); V1 > other four pooled p one-sided {q1:.1e}, two-sided {q2:.1e}")
    print(f"  cells where both motion models beat static: {n_beat} of {n_cells}")
    for m in MODELS:
        for k, lab in ((1, "(a) each area, one-sided"), (2, "(b) pooled, one-sided"), (3, "(a) each area, two-sided"), (4, "(b) pooled, two-sided")):
            n_ok = sum(v1_best[(s, m)][0] and v1_best[(s, m)][k] for s in SUBJECTS)
            print(f"  {m}, rank-sum reading {lab}: subjects with V1 best and p < 0.0001: {n_ok} of 3")

**Result: reproduced.** Both motion models beat the static model in all 15 area and subject cells under both test-signal readings. V1 has the highest mean in every subject for every model. With V1 against the other four areas pooled (rank-sum reading b), p is below 0.0001 in all three subjects for every model, one-sided or two-sided (largest 2.8e-10, S3 directional, two-sided). With V1 against each other area in turn (reading a), this holds in all three subjects for the static model only; the nondirectional model passes in 2 of 3 subjects and the directional model in 1 of 3 (V1 against V3A, directional, p = 4.1e-04 in S1 and 3.8e-02 in S3, one-sided).

## fig2fgh-retinotopic-receptive-fields

> Voxel receptive fields estimated from the motion-energy model are spatially localized and retinotopically organized; their angle maps are similar to those from conventional multifocal mapping, and their eccentricity maps are similar except in the far periphery, where the multifocal mapping stimulus was coarse.

Selectivity Estimation: each voxel's directional model (penalty reading a) is driven by dynamic Gaussian white noise in each cell of a 17 x 17 grid on uniform gray, and its predicted response minus the response to uniform gray gives a spatial map, to which a 2D Gaussian is fit (vim2.gaussian_fit). The multifocal maps are not in the data set, so the claims file's proxy test is used: in each subject, more than half of the retained voxels have a fit centered inside the 20 degree display; left-hemisphere V1 centers lie over 75% in the right visual field and the reverse; and eccentricity rises along V1 from the occipital pole forward (positive correlation, p < 0.05). Voxels are retained when the prediction accuracy is positive with p <= 0.01 (two-sided t test of the correlation) on (a) the 486 test samples or (b) the 750 held-out training samples. The posterior-anterior axis is the slice axis of the 64 x 64 x 18 volume (18 coronal slices, Experimental Procedures); its direction is taken from the data, as the direction from the V1 masks to the MT masks, which lie anterior to V1.

In [ ]:
rng_rf = np.random.default_rng([SEED, 20])
D = FEAT["directional"]
t = time.time()
noise = vim2.noise_probes(rng_rf)
F_noise = vim2.probe_features(noise, filters, D["mu"], D["sd"], D["cols"])
F_gray = vim2.probe_features([np.full_like(noise[0], vim2.GRAY)], filters, D["mu"], D["sd"], D["cols"])
movies, GLAB = vim2.grating_probes((0, 2, 4, 8, 16, 32), (0, 2, 4), range(0, 360, 45))
F_grat = vim2.probe_features(movies, filters, D["mu"], D["sd"], D["cols"])
del noise, movies
print(f"probe features in {time.time() - t:.0f} s: {len(F_noise)} noise cells, {len(F_grat)} gratings")
RF = {}
for s in SUBJECTS:
    f = FIT[s]["directional"]
    W = (D["X"].T @ f["C"]).reshape(len(vim2.DELAYS), len(D["cols"]), -1).sum(0)  # weights summed over delays
    maps = (F_noise - F_gray) @ W
    tune = ((F_grat - F_gray) @ W).reshape(6, 3, 8, -1).mean(2)  # (spatial freq, temporal freq, voxel)
    r0, c0, sig, fit_r = vim2.gaussian_fit(maps.T)
    vox = DATA[s]["vox"]
    p_test = vim2.corr_pvalue(f["acc_a"]["avg"], 486)
    p_held = vim2.corr_pvalue(f["held_acc"], int(held.sum()))
    RF[s] = dict(r0=r0, c0=c0, sig=sig, fit_r=fit_r, tune=tune,
                 keep={"a": (p_test <= 0.01) & (f["acc_a"]["avg"] > 0), "b": (p_held <= 0.01) & (f["held_acc"] > 0)},
                 inside=(r0 >= -0.5) & (r0 <= 16.5) & (c0 >= -0.5) & (c0 <= 16.5),
                 ecc=np.hypot(r0 - 8, c0 - 8) * 20 / 17, right=c0 > 8, z=vim2.voxel_xyz(vox)[2])
    del W, maps
print(f"receptive fields and tuning in {time.time() - t:.0f} s")

In [ ]:
DEG = 20 / 17
for s in SUBJECTS:
    r = RF[s]
    sl = DATA[s]["info"]["roi_mean_slice"]
    v1z = np.mean([sl["v1lh"], sl["v1rh"]])
    mtz = np.mean([sl[k] for k in ("MTlh", "MTrh") if k in sl])
    sign = 1 if mtz > v1z else -1  # +1: slice index grows anteriorly
    print(f"--- S{s}: mean slice of the V1 masks {v1z:.1f}, of the MT masks {mtz:.1f}; anterior is {'higher' if sign > 0 else 'lower'} slice index")
    for rd in ("a", "b"):
        k = r["keep"][rd]
        hemi, area = DATA[s]["hemi"], DATA[s]["area"]
        frac_in = r["inside"][k].mean()
        lh = k & r["inside"] & (area == "v1") & (hemi == "lh")
        rh = k & r["inside"] & (area == "v1") & (hemi == "rh")
        f_lh, f_rh = r["right"][lh].mean(), 1 - r["right"][rh].mean()
        v1 = k & r["inside"] & (area == "v1")
        rho, p = stats.pearsonr(r["ecc"][v1], sign * r["z"][v1])
        rho_s, p_s = stats.spearmanr(r["ecc"][v1], sign * r["z"][v1])
        ok = frac_in > 0.5 and f_lh > 0.75 and f_rh > 0.75 and rho > 0 and p < 0.05
        print(f"  p-value reading ({rd}): retained {k.sum()} of {len(k)} voxels; fit inside the display {frac_in:.2f}; "
              f"fit correlation median {np.median(r['fit_r'][k]):.2f}; Gaussian s.d. median {np.median(r['sig'][k]) * DEG:.1f} deg")
        print(f"     V1 lh centers in the right field {f_lh:.2f} (n = {lh.sum()}), V1 rh centers in the left field {f_rh:.2f} (n = {rh.sum()})")
        print(f"     V1 eccentricity against anterior position: Pearson r = {rho:.3f} (p = {p:.1e}), Spearman {rho_s:.3f} (p = {p_s:.1e}), n = {v1.sum()}; "
              f"eccentricity range {r['ecc'][v1].min():.1f} to {r['ecc'][v1].max():.1f} deg; all three conditions met {ok}")
    for ar in ("v2", "v3"):
        k = r["keep"]["a"] & r["inside"] & (DATA[s]["area"] == ar)
        f_lh = r["right"][k & (DATA[s]["hemi"] == "lh")].mean()
        f_rh = 1 - r["right"][k & (DATA[s]["hemi"] == "rh")].mean()
        print(f"  {ar} (reading a, for reference): lh centers in the right field {f_lh:.2f}, rh in the left {f_rh:.2f}")
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
for a, s in zip(ax, SUBJECTS):
    r = RF[s]
    k = r["keep"]["a"] & (DATA[s]["area"] == "v1")
    for h, col in (("lh", "tab:blue"), ("rh", "tab:red")):
        kk = k & (DATA[s]["hemi"] == h)
        a.scatter((r["c0"][kk] - 8) * DEG, (8 - r["r0"][kk]) * DEG, s=3, c=col, label=f"V1 {h}")
    a.add_patch(plt.Rectangle((-10, -10), 20, 20, fill=False))
    a.set_xlim(-20, 20); a.set_ylim(-20, 20); a.set_aspect("equal")
    a.set_title(f"S{s}: V1 receptive-field centers (deg)", fontsize=9)
    n_out = int((k & ((np.abs(r["c0"] - 8) * DEG > 20) | (np.abs(8 - r["r0"]) * DEG > 20))).sum())
    a.text(-19, -19, f"{n_out} outside the axes", fontsize=8)
    a.legend(fontsize=7, loc="upper right")
plt.show()

**Result: reproduced on the claims file's proxy test, without the multifocal maps.** In each subject 95% to 97% of the retained voxels (3,036, 3,866 and 3,358 under p-value reading a) have a Gaussian fit centered inside the display (bound: over half). The centers of left-hemisphere V1 voxels lie in the right visual field in 90%, 91% and 84% of cases, and those of right-hemisphere V1 voxels in the left field in 91%, 89% and 89% (bound: over 75%). V1 eccentricity rises from the occipital pole forward, Pearson r = 0.535, 0.607 and 0.330 (p = 2.3e-61, 9.2e-89 and 2.9e-27). Reading b of the p-value gives the same pass in every subject (contralateral fractions 0.86 to 0.92, r = 0.311 to 0.607). The similarity of the angle and eccentricity maps to multifocal mapping, which the statement also asserts, could not be tested: the mapping data are not in the data set and not public.

## fig2j-speed-eccentricity

> Optimal speed (peak temporal frequency over peak spatial frequency) correlates positively with eccentricity, so peripheral voxels prefer higher speeds, in V1, V2 and V3 of all three subjects (t test for correlation coefficient, p < 0.0001).

Tuning from the directional model's predicted responses to full-field drifting gratings at the basis set's spatial frequencies (0 to 32 cycles/image), temporal frequencies (0, 2, 4 Hz) and eight directions, averaged over directions, minus the response to uniform gray (Selectivity Estimation). Peaks: (a) the maximum of the joint map; (b) the maximum of each marginal curve. Speed is the peak temporal frequency over the peak spatial frequency in cycles per degree (cycles per image / 20). Voxels with prediction p > 0.01 (readings a and b as above) or optimal spatial frequency 0 are left out; eccentricity is that of the Gaussian fit, for fits centered inside the display.

In [ ]:
SF = np.array([0, 2, 4, 8, 16, 32]) / 20.0
TF = np.array([0, 2, 4])
res_j = {}
for pk in ("a", "b"):
    for prd in ("a", "b"):
        n_pos = n_sig = 0
        for s in SUBJECTS:
            r = RF[s]
            tune = r["tune"]
            if pk == "a":
                flat = tune.reshape(18, -1).argmax(0)
                isf, itf = flat // 3, flat % 3
            else:
                isf, itf = tune.mean(1).argmax(0), tune.mean(0).argmax(0)
            speed = np.where(isf > 0, TF[itf] / np.where(isf > 0, SF[isf], 1), np.nan)
            for ar in ("v1", "v2", "v3"):
                k = r["keep"][prd] & r["inside"] & (DATA[s]["area"] == ar) & (isf > 0)
                rho, p = stats.pearsonr(r["ecc"][k], speed[k])
                n_pos += rho > 0
                n_sig += (rho > 0) and (p < 1e-4)
                res_j[(pk, prd, s, ar)] = (rho, p, int(k.sum()))
                print(f"peaks ({pk}), p-value ({prd}), S{s} {ar}: n = {k.sum()}, r = {rho:.3f}, p = {p:.1e}; "
                      f"speed range {np.nanmin(speed[k]):.2f} to {np.nanmax(speed[k]):.2f} deg/s; dropped for spatial frequency 0: {int((r['keep'][prd] & r['inside'] & (DATA[s]['area'] == ar) & (isf == 0)).sum())}")
        print(f"  peaks ({pk}), p-value ({prd}): positive {n_pos} of 9, positive with p < 0.0001 {n_sig} of 9")
for s in SUBJECTS:
    tune = RF[s]["tune"]
    flat = tune.reshape(18, -1).argmax(0)
    print(f"S{s}: voxels by joint peak (spatial freq c/image, temporal freq Hz):",
          dict(collections.Counter((int(SF[i // 3] * 20), int(TF[i % 3])) for i in flat[RF[s]['keep']['a']]).most_common()))

**Result: reproduced.** Under every reading of the peaks (joint map or marginal curves) and of the p-value, optimal speed correlates positively with eccentricity in all nine area and subject cells with p < 0.0001: joint peaks and p-value reading a give r = 0.216 to 0.450 (largest p 8.4e-13, S2 V3); marginal peaks give r = 0.209 to 0.390 (largest p 2.8e-11). Voxels with an optimal spatial frequency of 0 are few (4 to 20 per cell) and left out.

## fig3ab-identification-s1

> For subject S1 the motion-energy model identifies the test second that evoked an observed BOLD pattern within one volume (1 s) 95% of the time (464 of 486 volumes), far above chance (under 1%, 3 of 486).

The 2,000 V1-V3B voxels whose model fit on 90% of the training data best predicts the held-out 10% are used (Voxel Selection). The likelihood is the multivariate Gaussian of Likelihood Estimation, with the covariance of the training residuals; the Tikhonov penalty is chosen from a grid of quarter decades by identification of the held-out 10% of the training data (claims file, Common methods), using the residuals of the 90% fit. Readings of the model: (a) four delays, (b) a single 4 s delay refit. Readings of the observed signal: (i) the mean of the ten repeats, (ii) each repeat identified on its own, the count averaged over repeats. Expected 464, tolerance 46.4 (at least 418).

In [ ]:
D1 = DATA[1]
Y, Yte, Yrep = D1["Y"], D1["Yte"], D1["Yrep"]
tt = np.arange(540)[kte]
th = np.arange(len(held))[held]
LAMS = np.round(np.logspace(-1, 2, 13), 3).tolist()
def within1(choice, times):
    return int(np.sum(np.abs(times[choice] - times) <= 1))
chance = float(np.mean([np.sum(np.abs(tt - t0) <= 1) for t0 in tt]))
print(f"chance count within one volume: {chance:.2f} of 486 ({100 * chance / 486:.2f}%)")
ident = {}
for mdl, key in (("a", "directional"), ("b", "delay4")):
    f = FIT[1][key]
    sel = np.argsort(f["held_acc"])[::-1][:2000]
    R90 = f["R90"]
    cov90 = (R90[:, sel].T @ R90[:, sel]) / R90.shape[0]
    hc = [within1(vim2.identify(Y[held][:, sel], f["Pheld"][:, sel], cov90, l), th) for l in LAMS]
    lam = LAMS[int(np.argmax(hc))]
    cov = (f["R"][:, sel].T @ f["R"][:, sel]) / f["R"].shape[0]
    res_lam = {}
    for l in LAMS:
        c_avg = within1(vim2.identify(Yte[:, sel], f["Pte"][:, sel], cov, l), tt)
        c_rep = [within1(vim2.identify(Yrep[k][:, sel], f["Pte"][:, sel], cov, l), tt) for k in range(10)]
        res_lam[l] = (c_avg, float(np.mean(c_rep)), min(c_rep), max(c_rep))
    ident[mdl] = (lam, res_lam)
    print(f"model reading ({mdl}): held-out accuracy of the 2,000th voxel {f['held_acc'][sel[-1]]:.3f}; "
          f"held-out identification within 1 s by Tikhonov penalty {dict(zip(LAMS, hc))} of {held.sum()}; chosen {lam}")
    for l, (ca, cr, lo, hi) in res_lam.items():
        mark = " <- chosen" if l == lam else ""
        print(f"   penalty {l}: (i) repeat mean {ca} of 486 ({100 * ca / 486:.1f}%); (ii) single repeats mean {cr:.1f} (range {lo} to {hi}){mark}")

**Result: partially reproduced.** With the four-delay directional model, the repeat-averaged test signal and the Tikhonov penalty chosen on the held-out training data (3.162 times the mean residual variance), 409 of 486 S1 test volumes (84.2%) are identified within one volume, 9 short of the tolerance (at least 418) and 55 short of the paper's 464; chance is 2.96 of 486 (0.61%). No penalty of the grid reaches 418 (the largest count is 409). Single repeats give 337.8 on average (321 to 364), and the single 4 s delay model 325 (repeat mean) and 242.5 (single repeats). Identification is far above chance under every reading, but below the paper's count; the effect is present and weaker. The penalty reading (b) of the encoding fits, which raised prediction accuracy in fig2de, was not run for identification.

## Blocked claims

- **fig3c-identification-large-set**, **fig4-reconstructions**, **fig4d-reconstruction-accuracy**: need the 1,000,000-clip identification set or the 18,000,000 s sampled prior of YouTube clips. The data set holds neither (the descriptor lists only the experiment's stimuli), and the paper gives no location for them (data_missing). The training movies (7,200 s) could serve as distractors for a partial test of fig3c, but not at the 1,000,000 clips the criterion names.

In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory {mem_gb():.1f} GB")

## Summary

| Claim | Outcome |
|---|---|
| fig2de-prediction-accuracy | reproduced under penalty reading b (0.233, 0.384, 0.386 against 0.24, 0.39, 0.40); under reading a the static mean, 0.211, is 0.005 outside tolerance; rests on ridge in place of L1 |
| fig2e-directional-gain-small | reproduced (gain 0.0010 to 0.0034 against 0.083 to 0.159) |
| figs1a-motion-better-every-area | reproduced (15 of 15 cells; V1 best in 3 of 3 subjects, pooled rank-sum) |
| fig2fgh-retinotopic-receptive-fields | reproduced on the proxy test (fits inside the display 95% to 97%; contralateral 84% to 91%; eccentricity rises forward in V1, r = 0.33 to 0.61) |
| fig2j-speed-eccentricity | reproduced (9 of 9 correlations positive with p < 0.0001 under every reading) |
| fig3ab-identification-s1 | partially reproduced (409 of 486 against 464; at least 418 needed) |
| fig3c-identification-large-set | blocked, data_missing |
| fig4-reconstructions | blocked, data_missing |
| fig4d-reconstruction-accuracy | blocked, data_missing |
| discussion-color-no-gain | reproduced in `color_no_gain.ipynb` (color model 0.009 lower) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** Ridge regression replaces the L1-regularized regression, with the penalty chosen per voxel on held-out training data under both readings of the claims file; an L1 fit was tried while preparing and did not converge in the time (not run here). The test repeats are z-scored per voxel before averaging, since the data set's test responses are not in the units of the training responses. The filter bank is pymoten's, whose default envelopes give the paper's 6,555 filters; its 855 zero-spatial-frequency filters have no direction and are kept as single features of the nondirectional model (4,275 features). Receptive fields and tuning are estimated with simulated probes whose contrast and duration the paper does not give (standard choices), and judged on the claims file's proxy test, since the multifocal maps are not public.

**What was found about the dataset.** The files match the descriptor in shape (7,200 and 540 samples, 73,728 voxels, 108,000 and 8,100 frames) and `rv` is the mean of `rva`. The descriptor does not say that the responses include the first 6 s of each block that the paper discards (the mean training response there is 0.28 to 0.42, against 0.01 elsewhere), that the test responses are on a different scale from the z-scored training responses (s.d. 0.10 to 0.11 for `rv`, 0.20 to 0.23 for `rva`, against 0.99 for `rt`), or that 29 to 179 V1-V3B voxels per subject hold missing values. The "peak BOLD responses" it mentions are not delay-shifted: the clock check gives 0.30 to 0.33 at a 4 s delay against 0.06 to 0.09 at 0 s. The descriptor names one ROI mask by example; the files hold many more (V1 to V4, V3A, V3B, MT and others, by hemisphere, different per subject), and one subject 3 voxel lies in two V1-V3B masks. The stored frames are upright once the last two stored axes are swapped (example frames above).